### Load data and do a quick check

In [1]:
import json
import pandas as pd
with open("../data/raw/batdongsan.txt", encoding="utf-8") as file:
    df = pd.DataFrame(json.load(file))

In [2]:
df.head(3)

,Loại hình nhà ở,Diện tích đất,Tổng số tầng,Giấy tờ pháp lý,address,price,city,Số phòng ngủ,Số phòng vệ sinh,Hướng ban công,Hướng cửa chính,Dự án,Tầng số
0,Nhà mặt tiền,65 m²(4.0x16.0),4,Sổ hồng,"Đường Nguyễn Trãi, Phường 7, Quận 5, TP.HCM",48.0,ho-chi-minh,NaN,NaN,NaN,NaN,NaN,NaN
1,Nhà hẻm ngõ,"105 m²(4,6x23,0)",2,Sổ hồng,"1234, Đường Huỳnh Tấn phát, Phường Tân Phú, Qu...",8.2,ho-chi-minh,6 phòng,6 WC,Đông,NaN,NaN,NaN
2,Biệt thự,"1.100 m²(20,0x50,0)",3,Sổ hồng,"105, Đường Trần Văn Kiểu, Phường 10, Quận 6, T...",300.0,ho-chi-minh,8 phòng,10 WC,NaN,NaN,NaN,NaN


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5414 entries, 0 to 5413
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Loại hình nhà ở   5414 non-null   object
 1   Diện tích đất     5414 non-null   object
 2   Tổng số tầng      3352 non-null   object
 3   Giấy tờ pháp lý   5414 non-null   object
 4   address           5414 non-null   object
 5   price             5414 non-null   object
 6   city              5414 non-null   object
 7   Số phòng ngủ      5008 non-null   object
 8   Số phòng vệ sinh  4631 non-null   object
 9   Hướng ban công    463 non-null    object
 10  Hướng cửa chính   2804 non-null   object
 11  Dự án             110 non-null    object
 12  Tầng số           1 non-null      object
dtypes: object(13)
memory usage: 550.0+ KB


### Basic cleaning (before merging and more preprocessing)
Rename columns

In [3]:
rename_map = {
    "Loại hình nhà ở":"property_type",
    "Diện tích đất":"area",
    "Tổng số tầng":"n_floors",
    "Giấy tờ pháp lý":"legal_docs",
    "Số phòng ngủ":"n_bedrooms",
    "Số phòng vệ sinh":"n_bathrooms",
    "Hướng ban công":"balcony_direction",
    "Hướng cửa chính":"facing_direction",
    "Dự án":"project",
    "Tầng số":"floor_num",
    "city":"city_province"
}
df.rename(rename_map, axis=1, inplace=True)
df.head(2)

,property_type,area,n_floors,legal_docs,address,price,city_province,n_bedrooms,n_bathrooms,balcony_direction,facing_direction,project,floor_num
0,Nhà mặt tiền,65 m²(4.0x16.0),4,Sổ hồng,"Đường Nguyễn Trãi, Phường 7, Quận 5, TP.HCM",48.0,ho-chi-minh,NaN,NaN,NaN,NaN,NaN,NaN
1,Nhà hẻm ngõ,"105 m²(4,6x23,0)",2,Sổ hồng,"1234, Đường Huỳnh Tấn phát, Phường Tân Phú, Qu...",8.2,ho-chi-minh,6 phòng,6 WC,Đông,NaN,NaN,NaN


Extract the numeric values for numeric columns

In [4]:
import re

def extract_numeric(s:str|None, thousands_sep:bool=True) -> float:
    if not isinstance(s,str) or not s:       # Handle np.nan (a float), or empty strings
        return None                        # Return np.nan cuz None is treated like a value

    results = re.search(r"^(\d+.?\d*,?\d*)\D?", s)
    if not results:
        return None

    num_str = results.group(1)
    if thousands_sep:
        num_str = num_str.replace(".","")
    num_str = num_str.replace(",",".")

    return float(num_str)
    
def extract_measuring_unit(s:str|None) -> str:
    if not isinstance(s,str) or not s:
        return None

    results = re.search(r"^\d+.?\d*,?\d*\s*(\D*)\(*", s)
    if not results:
        return None
    else:
        return results.group(1).replace('(', '').strip()

def extract_dimensions(area_raw:str):
    if not isinstance(area_raw,str) or not area_raw:
        return None

    results = re.search(r"\((.*)x(.*)\)", area_raw)
    if not results:
        return pd.Series((None, None))
    return pd.Series((
        extract_numeric(results.group(1), thousands_sep=False), 
        extract_numeric(results.group(2), thousands_sep=False)
    ))

In [5]:
df[['dimension_1', 'dimension_2']] = df['area'].apply(extract_dimensions)
df['area_num'] = df['area'].apply(extract_numeric)
df['area_unit'] = df['area'].apply(extract_measuring_unit)
df['n_bedrooms_2'] = df['n_bedrooms'].apply(extract_numeric)
df['n_bathrooms_2'] = df['n_bathrooms'].apply(extract_numeric)
df['price_2'] = pd.to_numeric(df['price'], errors='coerce', downcast='float')
df['n_floors_2'] = pd.to_numeric(df['n_floors'], errors='coerce')

df[['area', 'area_num', 'area_unit', 'dimension_1', 'dimension_2', 
    'n_bedrooms', 'n_bedrooms_2', 'n_bathrooms', 'n_bathrooms_2', 
    'price', 'price_2', 
    'n_floors', 'n_floors_2']].sample(10)

,area,area_num,area_unit,dimension_1,dimension_2,n_bedrooms,n_bedrooms_2,n_bathrooms,n_bathrooms_2,price,price_2,n_floors,n_floors_2
735,"25,4 m²",25.4,m²,NaN,NaN,3 phòng,3.0,2 WC,2.0,3.5,3.50,3,3.0
3299,"135 m²(5,5x25,0)",135.0,m²,5.5,25.0,3 phòng,3.0,4 WC,4.0,19.5,19.50,3,3.0
2156,"100 m²(5,0x20,0)",100.0,m²,5.0,20.0,3 phòng,3.0,3 WC,3.0,1.25,1.25,NaN,NaN
2454,1.200 m²,1200.0,m²,NaN,NaN,1 phòng,1.0,1 WC,1.0,4.5,4.50,1,1.0
2069,"90 m²(5,0x18,0)",90.0,m²,5.0,18.0,3 phòng,3.0,2 WC,2.0,0.6,0.60,2,2.0
563,80 m²(5.0x16.0),80.0,m²,5.0,16.0,3 phòng,3.0,NaN,NaN,22.0,22.00,NaN,NaN
813,"45 m²(5,0x9,0)",45.0,m²,5.0,9.0,3 phòng,3.0,2 WC,2.0,4.0,4.00,3,3.0
991,"82 m²(7,0x12,0)",82.0,m²,7.0,12.0,NaN,NaN,NaN,NaN,12.5,12.50,4,4.0
4930,"5.000 m²(50,0x92,0)",5000.0,m²,50.0,92.0,NaN,NaN,NaN,NaN,630.0,630.00,NaN,NaN
1265,"51 m²(4,3x13,0)",51.0,m²,4.3,13.0,2 phòng,2.0,2 WC,2.0,5.8,5.80,1,1.0


In [7]:
df[['area', 'area_num', 'dimension_1', 'dimension_2', 
    'n_bedrooms', 'n_bedrooms_2', 'n_bathrooms', 'n_bathrooms_2', 
    'price', 'price_2', 
    'n_floors', 'n_floors_2'
]].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5414 entries, 0 to 5413
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   area           5414 non-null   object 
 1   area_num       5414 non-null   float64
 2   dimension_1    4692 non-null   float64
 3   dimension_2    4692 non-null   float64
 4   n_bedrooms     5008 non-null   object 
 5   n_bedrooms_2   5008 non-null   float64
 6   n_bathrooms    4631 non-null   object 
 7   n_bathrooms_2  4631 non-null   float64
 8   price          5414 non-null   object 
 9   price_2        5406 non-null   float32
 10  n_floors       3352 non-null   object 
 11  n_floors_2     3352 non-null   float64
dtypes: float32(1), float64(6), object(5)
memory usage: 486.5+ KB


In [38]:
df.loc[df.price_2.isna()]

,property_type,area,n_floors,legal_docs,address,price,city_province,n_bedrooms,n_bathrooms,balcony_direction,facing_direction,project,floor_num,dimension_1,dimension_2,area_num,n_bedrooms_2,n_bathrooms_2,price_2,n_floors_2
13,Nhà mặt tiền,40 m²,2,Sổ đỏ,"Đường Nguyễn Công Trứ, Phường 19, Quận Bình Th...",thỏa thuận,ho-chi-minh,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.0,NaN,NaN,NaN,2.0
97,Nhà mặt tiền,"49,7 m²(4,0x12,0)",NaN,Sổ hồng,"Đường Nguyễn Tư Giản, Phường 12, Quận Gò Vấp, ...",thỏa thuận,ho-chi-minh,3 phòng,1 WC,NaN,NaN,NaN,NaN,4.0,12.0,49.7,3.0,1.0,NaN,NaN
280,Nhà hẻm ngõ,"315 m²(8,5x16,4)",2,Sổ hồng,"254/98/37 - 39, Đường Âu Cơ, Phường 9, Quận Tâ...",thỏa thuận,ho-chi-minh,NaN,NaN,NaN,NaN,NaN,NaN,8.5,16.4,315.0,NaN,NaN,NaN,2.0
1023,Nhà mặt tiền,80 m²,NaN,Sổ đỏ,"Đường Giảng Võ, Phường Giảng Võ, Quận Ba Đình,...",thỏa thuận,ha-noi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80.0,NaN,NaN,NaN,NaN
1102,Nhà hẻm ngõ,43 m²,NaN,Sổ đỏ,"Đường xóm 2, Xã Vĩnh Quỳnh, Huyện Thanh Trì, H...",thỏa thuận,ha-noi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,43.0,NaN,NaN,NaN,NaN
1114,Nhà hẻm ngõ,80 m²,5,Sổ đỏ,"Đường Lạc Long Quân, Phường Xuân La, Quận Tây ...",thỏa thuận,ha-noi,7 phòng,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80.0,7.0,NaN,NaN,5.0
4026,Biệt thự,"56 m²(4,0x14,0)",1,Sổ hồng,"Đường ĐT 719B, Xã Hàm Mỹ, Huyện Hàm Thuận Nam,...",thỏa thuận,binh-thuan,2 phòng,2 WC,NaN,Tây,NaN,NaN,4.0,14.0,56.0,2.0,2.0,NaN,1.0
5405,Biệt thự,180 m²,2,Sổ đỏ,"Thị Xã Phúc Yên, Vĩnh Phúc",thỏa thuận,vinh-phuc,NaN,NaN,NaN,NaN,Flamingo Resort Đại Lải,NaN,NaN,NaN,180.0,NaN,NaN,NaN,2.0


Observation:
- Non-null counts of `price_2` is lower than the original `price`: "Negotiable" prices 

In [39]:
df.legal_docs.unique()
# df.loc[df.legal_docs == 'Giấy tờ khác'] = 'Khác'

array(['Sổ hồng', 'Sổ đỏ', 'Giấy tờ hợp lệ', 'Giấy tờ khác',
       'Hợp đồng mua bán', 'Đang chờ sổ'], dtype=object)

Extract city and district

In [6]:
def extract_location_detail(addr: str, level:int) -> str|None:
    '''
    level:int - the level of details to extract from the address. 1 -> Province; 2 -> District
    '''
    if not isinstance(addr, str) or not addr.strip():
        return None
    parts = [p.strip() for p in addr.split(",") if p.strip() != ""]
    if len(parts) >= level:
        return parts[-level]
    return None

In [7]:
df['city_province'] = df.address.apply(extract_location_detail, level = 1)
df['district'] = df.address.apply(extract_location_detail, level = 2)
df[['address', 'city_province', 'district']]

,address,city_province,district
0,"Đường Nguyễn Trãi, Phường 7, Quận 5, TP.HCM",TP.HCM,Quận 5
1,"1234, Đường Huỳnh Tấn phát, Phường Tân Phú, Qu...",TP.HCM,Quận 7
2,"105, Đường Trần Văn Kiểu, Phường 10, Quận 6, T...",TP.HCM,Quận 6
3,"Đường Đoàn Nguyễn Tuấn, Xã Hưng Long, Huyện Bì...",TP.HCM,Huyện Bình Chánh
4,"Đường 3 Tháng 2, Phường 14, Quận 10, TP.HCM",TP.HCM,Quận 10
...,...,...,...
5409,"Xã Ngọc Thanh, Thị Xã Phúc Yên, Vĩnh Phúc",Vĩnh Phúc,Thị Xã Phúc Yên
5410,"436, Đường Tôn Đức Thắng, Xã Gia Khánh, Huyện ...",Vĩnh Phúc,Huyện Bình Xuyên
5411,"Phường Hùng Vương, Thị Xã Phúc Yên, Vĩnh Phúc",Vĩnh Phúc,Thị Xã Phúc Yên
5412,"Đường Yên Ninh, Phường Yên Ninh, TP. Yên Bái, ...",Yên Bái,TP. Yên Bái


### Export the file with extracted features

In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5414 entries, 0 to 5413
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   property_type      5414 non-null   object 
 1   area               5414 non-null   object 
 2   n_floors           3352 non-null   object 
 3   legal_docs         5414 non-null   object 
 4   address            5414 non-null   object 
 5   price              5414 non-null   object 
 6   city_province      5414 non-null   object 
 7   n_bedrooms         5008 non-null   object 
 8   n_bathrooms        4631 non-null   object 
 9   balcony_direction  463 non-null    object 
 10  facing_direction   2804 non-null   object 
 11  project            110 non-null    object 
 12  floor_num          1 non-null      object 
 13  dimension_1        4692 non-null   float64
 14  dimension_2        4692 non-null   float64
 15  area_num           5414 non-null   float64
 16  n_bedrooms_2       5008 

In [9]:
df_final = df[['property_type', 'price_2', 'area_num', 'area_unit', 'n_bedrooms_2', 'n_bathrooms_2',
                'n_floors_2', 'dimension_2', 'address', 'city_province', 'district','legal_docs', 'facing_direction', 'dimension_1']]
df_final = df_final.rename({
    'price_2': 'price',
    'area_num': 'area',
    'n_bedrooms_2': 'n_bedrooms',
    'n_bathrooms_2': 'n_bathrooms', 
    'dimension_1': 'front_width',
    'n_floors_2': 'n_floors',
}, axis=1)

In [10]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5414 entries, 0 to 5413
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   property_type     5414 non-null   object 
 1   price             5406 non-null   float32
 2   area              5414 non-null   float64
 3   area_unit         5414 non-null   object 
 4   n_bedrooms        5008 non-null   float64
 5   n_bathrooms       4631 non-null   float64
 6   n_floors          3352 non-null   float64
 7   dimension_2       4692 non-null   float64
 8   address           5414 non-null   object 
 9   city_province     5414 non-null   object 
 10  district          5414 non-null   object 
 11  legal_docs        5414 non-null   object 
 12  facing_direction  2804 non-null   object 
 13  front_width       4692 non-null   float64
dtypes: float32(1), float64(6), object(7)
memory usage: 571.1+ KB


In [11]:
df_final.sample(2)

,property_type,price,area,area_unit,n_bedrooms,n_bathrooms,n_floors,dimension_2,address,city_province,district,legal_docs,facing_direction,front_width
3887,Nhà hẻm ngõ,1.5,83.0,m²,2.0,2.0,2.0,14.0,"Phường An Lộc, Thị xã Bình Long, Bình Phước",Bình Phước,Thị xã Bình Long,Giấy tờ hợp lệ,Nam,6.0
4870,Biệt thự,630.0,5000.0,m²,NaN,NaN,NaN,92.0,"Đường Hoàng Quốc Việt, Bãi Cháy, TP. Hạ Long, ...",Quảng Ninh,TP. Hạ Long,Sổ đỏ,NaN,50.0


In [12]:
df_final.to_csv('../data/interim/muaban_net.csv')

In [14]:
df_final.city_province.unique()

array(['TP.HCM', 'Hà Nội', 'Đà Nẵng', 'Hải Phòng', 'Cần Thơ', 'Đồng Nai',
       'Bình Dương', 'Long An', 'Lâm Đồng', 'Bà Rịa - Vũng Tàu',
       'Đắk Lắk', 'Tiền Giang', 'Bắc Ninh', 'Quảng Nam', 'Khánh Hòa',
       'Nghệ An', 'An Giang', 'Bạc Liêu', 'Bắc Giang', 'Bến Tre',
       'Bình Định', 'Bình Phước', 'Bình Thuận', 'Cà Mau', 'Đồng Tháp',
       'Gia Lai', 'Hậu Giang', 'Hưng Yên', 'Kiên Giang', 'Kon Tum',
       'Lào Cai', 'Nam Định', 'Ninh Thuận', 'Phú Thọ', 'Phú Yên',
       'Quảng Bình', 'Quảng Ngãi', 'Quảng Ninh', 'Sóc Trăng', 'Tây Ninh',
       'Thái Bình', 'Thái Nguyên', 'Thanh Hóa', 'Thừa Thiên Huế',
       'Trà Vinh', 'Vĩnh Long', 'Vĩnh Phúc', 'Yên Bái'], dtype=object)